# 👁️ Day 3 — Computer Vision & Model Training
### AI Workshop · Google Colab Notebook

---

**Today you will:**
- Run a pre-trained YOLO model and see real-time object detection
- Annotate your own image dataset using Roboflow
- Train a custom YOLOv8 object detector on YOUR data
- Evaluate your model: mAP, precision, recall, confusion matrix
- Run inference on new images — see your model detect your objects

**The pipeline today:**
```
Your Images (DATA) → Annotate → Roboflow Dataset
                                      ↓
                              YOLOv8 Training (MODEL)
                                      ↓
                         Bounding Boxes + Labels (OUTPUT)
                                      ↓
                         App / Pipeline / Alert (INTEGRATION)
```

> ⚠️ **Before running anything:** Go to **Runtime → Change runtime type → T4 GPU → Save**
> Training on CPU will take 10x longer.

> 📋 **Run cells in order.** If you restart the runtime, re-run from Section 0.

---
## Section 0 — Setup
Install libraries and verify GPU. Run this first.

In [ ]:
# Install YOLOv8 (ultralytics) and Roboflow SDK
!pip install ultralytics roboflow -q

import torch
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np
import requests
from io import BytesIO
import os, shutil
import warnings # Added this line

# Suppress all warnings. Added this line
warnings.filterwarnings('ignore')

# Check GPU
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU detected: {gpu} ({mem:.1f} GB VRAM)')
else:
    print('⚠️  No GPU found — training will be slow!')
    print('   Go to: Runtime → Change runtime type → T4 GPU → Save')

print('\n✅ All packages ready.')

---
## Section 1 — 🔵 Demo: Pre-Trained YOLOv8

Before training your own model, let's see what a model trained on 80 million images can do.

**YOLOv8** knows 80 categories from the COCO dataset:
person, car, bicycle, dog, cat, chair, laptop, phone, bottle, and 71 more.

This is what **transfer learning** starts from.

In [ ]:
# Load the pre-trained YOLOv8 nano model
# 'n' = nano (fastest, smallest) — good for demos
# Other sizes: s (small), m (medium), l (large), x (extra-large)
print('Loading YOLOv8n (pre-trained on COCO)...')
pretrained_model = YOLO('yolov8n.pt')
print('✅ Model loaded.')
print(f'   Classes it knows: {len(pretrained_model.names)} categories')
print(f'   First 10: {list(pretrained_model.names.values())[:10]}')

In [ ]:
# ── HELPER: display detection results ────────────────────────────────────────
def show_detections(results, title='YOLOv8 Detections', figsize=(10, 8)):
    """
    Display YOLO detection results with bounding boxes, labels, confidence.
    """
    result   = results[0]
    img_plot = result.plot()          # BGR array with boxes drawn
    img_rgb  = img_plot[:, :, ::-1]  # convert BGR → RGB

    plt.figure(figsize=figsize)
    plt.imshow(img_rgb)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # Print text summary
    boxes = result.boxes
    if len(boxes) == 0:
        print('No objects detected.')
        return

    print(f'\n📊 Detected {len(boxes)} object(s):')
    print('-' * 40)
    names = result.names
    for box in boxes:
        cls  = int(box.cls[0])
        conf = float(box.conf[0])
        xyxy = box.xyxy[0].tolist()
        print(f'  {names[cls]:<20} confidence: {conf:.2%}')

print('Helper function ready.')

In [ ]:
# ── DEMO: Run detection on a sample street scene ─────────────────────────────
demo_url = 'https://ultralytics.com/images/bus.jpg'

print('Running YOLOv8 on a street scene...')
results = pretrained_model.predict(
    source     = demo_url,
    conf       = 0.25,      # confidence threshold (0–1)
    iou        = 0.45,      # IoU threshold for NMS
    verbose    = False,
)
show_detections(results, title='Pre-trained YOLOv8 — Street Scene')

In [ ]:
# 🟡 YOUR TURN — test on your own image URL
# ─────────────────────────────────────────────────────────────────────────────

your_image_url = 'PASTE YOUR IMAGE URL HERE'

# ─────────────────────────────────────────────────────────────────────────────

if 'PASTE' not in your_image_url:
    results = pretrained_model.predict(source=your_image_url, conf=0.25, verbose=False)
    show_detections(results, title='Your Image — Pre-trained YOLOv8')
else:
    print('⚠️  Replace the URL above with a real image URL, then run this cell.')

In [ ]:
# ── What the model DOESN'T know ──────────────────────────────────────────────
# Try detecting something NOT in COCO — watch it fail or misclassify
# This motivates why we train a custom model

uncommon_url = 'https://d3i6fh83elv35t.cloudfront.net/static/2018/11/RTS27MFD-1024x692.jpg'

print('Testing on chess pieces (not in COCO dataset)...')
results = pretrained_model.predict(source=uncommon_url, conf=0.20, verbose=False)
show_detections(results, title='Chess Pieces — What does YOLOv8 see?')

print('\n🤔 The model may misclassify or miss chess pieces entirely.')
print('   This is exactly why we train custom models on our own data.')

---
## Section 2 — Understanding What YOLO Sees

Before training, let's build intuition for how the model represents what it detects.

In [ ]:
# ── What does a detection actually contain? ───────────────────────────────────
# Run detection and inspect the raw output structure

results = pretrained_model.predict(
    source  = 'https://ultralytics.com/images/bus.jpg',
    conf    = 0.25,
    verbose = False
)
result = results[0]

print('🔍 Raw detection output structure:')
print('=' * 50)
print(f'Image size:      {result.orig_shape}')
print(f'Number of boxes: {len(result.boxes)}')
print()

print('Each detection contains:')
for i, box in enumerate(result.boxes[:3]):
    cls_id = int(box.cls[0])
    conf   = float(box.conf[0])
    xyxy   = [round(v, 1) for v in box.xyxy[0].tolist()]  # pixel coords
    xywhn  = [round(v, 4) for v in box.xywhn[0].tolist()] # normalised 0-1

    print(f'\n  Detection {i+1}: {result.names[cls_id]}')
    print(f'    Class ID:      {cls_id}')
    print(f'    Confidence:    {conf:.2%}')
    print(f'    Bounding box:  x1={xyxy[0]} y1={xyxy[1]} x2={xyxy[2]} y2={xyxy[3]} (pixels)')
    print(f'    Normalised:    cx={xywhn[0]} cy={xywhn[1]} w={xywhn[2]} h={xywhn[3]} (0–1 scale)')

print('\n💡 The normalised format (0–1) is what YOLO uses internally.')
print('   It makes the model resolution-independent.')

In [ ]:
# ── Confidence threshold demo ─────────────────────────────────────────────────
# Show how changing the threshold changes what gets detected

test_url  = 'https://ultralytics.com/images/bus.jpg'
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Effect of Confidence Threshold', fontsize=14, fontweight='bold')

for ax, threshold in zip(axes, [0.1, 0.4, 0.75]):
    res    = pretrained_model.predict(source=test_url, conf=threshold, verbose=False)
    img    = res[0].plot()[:, :, ::-1]
    n_det  = len(res[0].boxes)
    ax.imshow(img)
    ax.set_title(f'conf = {threshold}  →  {n_det} detections', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

print('🤔 Which threshold is "right"?')
print('   It depends on the application:')
print('   Low threshold  → catch everything (more false positives)')
print('   High threshold → only confident detections (may miss real objects)')
print('   This is a design decision, not a model decision.')

---
## Section 3 — 🟡 Training Your Custom Model

Now we train on YOUR dataset from Roboflow.

**Steps before running this section:**
1. Go to [roboflow.com](https://roboflow.com) and open your project
2. Click **Generate** → create a dataset version
3. Click **Export** → choose **YOLOv8** format → **Show download code**
4. Copy the code snippet — you'll paste it in the cell below

> 💡 **Don't have your own dataset yet?** Use the backup dataset in Section 3B.

In [ ]:
# ── 3A: Download YOUR Roboflow dataset ───────────────────────────────────────
# Paste the code from Roboflow's export panel below
# It will look like this (replace with your actual values):
# ─────────────────────────────────────────────────────────────────────────────

# from roboflow import Roboflow

# # PASTE YOUR ROBOFLOW EXPORT CODE HERE:
# rf      = Roboflow(api_key='YOUR_API_KEY_HERE')
# project = rf.workspace('YOUR_WORKSPACE').project('YOUR_PROJECT')
# version = project.version(1)
# dataset = version.download('yolov8')

# # ─────────────────────────────────────────────────────────────────────────────

# # The dataset downloads to a folder — find the data.yaml path
# data_yaml = dataset.location + '/data.yaml'
# print(f'\n✅ Dataset downloaded to: {dataset.location}')
# print(f'   data.yaml path: {data_yaml}')

# # Verify the structure
# import yaml
# with open(data_yaml) as f:
#     info = yaml.safe_load(f)
# print(f"\n📊 Dataset info:")
# print(f"   Classes: {info.get('names', [])}")
# print(f"   Number of classes: {info.get('nc', '?')}")

In [ ]:
# ── 3B: BACKUP — use a pre-built dataset (if you don't have your own yet) ────
# This downloads a fruit detection dataset: apple, banana, orange
# Comment out Section 3A and run this instead

# Uncomment ALL lines below to use the backup dataset:

from roboflow import Roboflow
from google.colab import userdata
rf      = Roboflow(api_key='')  # still need YOUR api key
project = rf.workspace("myworkspace-xe3ef").project("fruit-detection-gu1qj")
version = project.version(1)
dataset = version.download("yolov8")


print('If using backup dataset: uncomment the lines above and run.')
print('If using your own dataset: use Section 3A above.')

In [ ]:
# ── Inspect your dataset before training ─────────────────────────────────────
# Always look at your data before training — catch problems early

import glob
from pathlib import Path

dataset_path = Path(dataset.location)

# Count images and labels
train_imgs   = list((dataset_path / 'train' / 'images').glob('*.jpg')) + \
               list((dataset_path / 'train' / 'images').glob('*.png'))
val_imgs     = list((dataset_path / 'valid' / 'images').glob('*.jpg')) + \
               list((dataset_path / 'valid' / 'images').glob('*.png'))

print('📊 Dataset Summary')
print('=' * 40)
print(f'Training images:   {len(train_imgs)}')
print(f'Validation images: {len(val_imgs)}')

# Show a few sample images with annotations
sample_imgs = train_imgs[:4]
fig, axes   = plt.subplots(1, min(4, len(sample_imgs)), figsize=(16, 4))
if len(sample_imgs) == 1:
    axes = [axes]

for ax, img_path in zip(axes, sample_imgs):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(img_path.name[:20], fontsize=9)
    ax.axis('off')

    # Draw annotations if label file exists
    label_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
    if label_path.exists():
        w, h = img.size
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    _, cx, cy, bw, bh = map(float, parts)
                    x1 = (cx - bw/2) * w
                    y1 = (cy - bh/2) * h
                    rect = patches.Rectangle(
                        (x1, y1), bw*w, bh*h,
                        linewidth=2, edgecolor='lime', facecolor='none'
                    )
                    ax.add_patch(rect)

plt.suptitle('Sample Training Images with Annotations', fontsize=12)
plt.tight_layout()
plt.show()

print('\n🔍 Check: Are the green boxes tight around the objects?')
print('   If boxes are sloppy or missing, fix annotations in Roboflow first.')

In [ ]:
# ── TRAIN YOUR MODEL ──────────────────────────────────────────────────────────
# This is the main event. Training takes 5–15 minutes on a T4 GPU.
# Watch the loss columns scroll — both should decrease over epochs.
#
# What the columns mean:
#   box_loss  = error in bounding box position/size prediction
#   cls_loss  = error in class label prediction
#   dfl_loss  = distribution focal loss (fine-grained box regression)
#   Precision = of detections made, % that were correct
#   Recall    = of real objects, % that were found
#   mAP50     = mean Average Precision at 50% IoU threshold
#   mAP50-95  = mAP averaged across IoU thresholds 50%→95% (harder)

print('🚀 Starting training...')
print('   This will take 5–15 minutes. Watch the loss columns go down.')
print('   Both box_loss and cls_loss should decrease each epoch.\n')

custom_model = YOLO('yolov8n.pt')   # start from COCO pre-trained weights

results = custom_model.train(
    data     = '/content/fruit-detection-1/data.yaml',   # path to your dataset config
    epochs   = 30,          # number of full passes through the dataset
    imgsz    = 640,         # input image size (must match Roboflow export)
    batch    = 16,          # images per gradient step (reduce to 8 if OOM)
    patience = 10,          # stop early if no improvement for 10 epochs
    device   = 0 if torch.cuda.is_available() else 'cpu',
    project  = 'workshop_runs',
    name     = 'my_detector',
    exist_ok = True,
)

print('\n✅ Training complete!')
print(f'   Best model saved to: workshop_runs/my_detector/weights/best.pt')

---
## Section 4 — Evaluating Your Model

Training finished. Now we understand *how well* it worked and *why*.

In [ ]:
# ── Load your best trained model ──────────────────────────────────────────────
best_model_path = '/content/runs/detect/workshop_runs/my_detector/weights/best.pt'

trained_model = YOLO(best_model_path)
print(f'✅ Best model loaded from: {best_model_path}')
print(f'   Classes: {trained_model.names}')

In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────────
# These show how loss and metrics changed across epochs

results_csv = '/content/runs/detect/workshop_runs/my_detector/results.csv'

if os.path.exists(results_csv):
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()   # remove whitespace from col names

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('Training Curves — Your Custom Model', fontsize=14, fontweight='bold')

    plots = [
        ('train/box_loss', 'Box Loss (train)', 'Should decrease'),
        ('train/cls_loss', 'Class Loss (train)', 'Should decrease'),
        ('val/box_loss',   'Box Loss (val)',   'Should decrease'),
        ('metrics/precision(B)', 'Precision', 'Should increase'),
        ('metrics/recall(B)',    'Recall',    'Should increase'),
        ('metrics/mAP50(B)',     'mAP@50',    'Should increase'),
    ]

    colors = ['#E74C3C','#E67E22','#E74C3C','#2ECC71','#3498DB','#9B59B6']

    for ax, (col, title, hint), color in zip(axes.flat, plots, colors):
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color=color, linewidth=2)
            ax.set_title(f'{title}\n({hint})', fontsize=10)
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, f'Column\n{col}\nnot found',
                    ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title)

    plt.tight_layout()
    plt.show()

    # Final metrics summary
    last = df.iloc[-1]
    print('\n📊 Final Metrics (last epoch):')
    print('=' * 40)
    for col, label, _ in plots:
        if col in df.columns:
            print(f'  {label:<25} {last[col]:.4f}')
else:
    print('results.csv not found — run training first (Section 3)')

In [ ]:
# ── Run validation ────────────────────────────────────────────────────────────
# Evaluate the model on the validation set

print('Running validation on held-out images...')
val_metrics = trained_model.val(
    data    = '/content/fruit-detection-1/data.yaml',
    imgsz   = 640,
    verbose = True,
)

print('\n📊 Validation Summary:')
print('=' * 40)
print(f'  mAP@50:    {val_metrics.box.map50:.4f}')
print(f'  mAP@50-95: {val_metrics.box.map:.4f}')
print(f'  Precision: {val_metrics.box.mp:.4f}')
print(f'  Recall:    {val_metrics.box.mr:.4f}')

print('\n💡 Interpreting mAP@50:')
print('   < 0.30  → Needs more data or better annotations')
print('   0.30–0.50 → Decent start — more data and epochs will help')
print('   0.50–0.75 → Good — production-ready for many applications')
print('   > 0.75  → Excellent — visually distinct classes, clean labels')

In [ ]:
# ── Show confusion matrix and PR curve ───────────────────────────────────────
# These are saved automatically during training

run_dir = '/content/runs/detect/workshop_runs/my_detector/'

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

conf_matrix_path = run_dir + 'confusion_matrix_normalized.png'
pr_curve_path    = run_dir + 'PR_curve.png'

for ax, path, title in [
    (axes[0], conf_matrix_path, 'Confusion Matrix (normalised)'),
    (axes[1], pr_curve_path,    'Precision-Recall Curve'),
]:
    if os.path.exists(path):
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.axis('off')
    else:
        ax.text(0.5, 0.5, f'{title}\nnot found yet',
                ha='center', va='center', fontsize=12, transform=ax.transAxes)
        ax.set_title(title)

plt.tight_layout()
plt.show()

print('Confusion Matrix tells you: what does the model confuse with what?')
print('PR Curve tells you: how precision and recall trade off at different thresholds.')

---
## Section 5 — 🟡 Run Inference on Your Own Images

Your model is trained. Now test it on images it has never seen.

In [ ]:
# ── Inference on an image URL ─────────────────────────────────────────────────
# Replace with an image of your target class

test_url = 'https://nutritionsource.hsph.harvard.edu/wp-content/uploads/2018/08/bananas-1354785_1920.jpg'

if 'PASTE' not in test_url:
    results = trained_model.predict(
        source  = test_url,
        conf    = 0.25,
        verbose = False,
    )
    show_detections(results, title='Your Custom Model — Test Inference')
else:
    print('⚠️  Replace the URL with an image of your target object.')

In [ ]:
# ── Inference on an uploaded image ───────────────────────────────────────────
# Upload an image from your phone or computer
# Files panel on left → Upload → then run this cell

from google.colab import files
import io

print('https://assets.epicurious.com/photos/68557ec0c13b32a411b3972b/16:9/w_1920%2Cc_limit/types-of-apples_LEDE_V3_061025_3678_VOG_final.jpg')
uploaded = files.upload()

for filename, content in uploaded.items():
    img = Image.open(io.BytesIO(content))
    img.save(f'/content/{filename}')

    results = trained_model.predict(
        source  = f'/content/{filename}',
        conf    = 0.25,
        verbose = False,
    )
    show_detections(results, title=f'Your Custom Model → {filename}')

In [ ]:
# ── Threshold experiment ──────────────────────────────────────────────────────
# Find the threshold where your model starts making mistakes

test_url = 'https://assets.epicurious.com/photos/68557ec0c13b32a411b3972b/16:9/w_1920%2Cc_limit/types-of-apples_LEDE_V3_061025_3678_VOG_final.jpg'

if 'PASTE' not in test_url:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle('Your Model — Confidence Threshold Experiment', fontsize=13)

    for ax, thresh in zip(axes, [0.1, 0.3, 0.5, 0.75]):
        res   = trained_model.predict(source=test_url, conf=thresh, verbose=False)
        img   = res[0].plot()[:, :, ::-1]
        n_det = len(res[0].boxes)
        ax.imshow(img)
        ax.set_title(f'conf={thresh}  →  {n_det} detections')
        ax.axis('off')

    plt.tight_layout()
    plt.show()
    print('\n🤔 At what threshold does the model start making false detections?')
    print('   At what threshold does it start missing real objects?')
else:
    print('⚠️  Paste your test image URL above first.')

---
## Section 6 — Compare: Pre-trained vs Your Model

In [ ]:
# ── Side-by-side comparison ───────────────────────────────────────────────────
# Pre-trained COCO model vs YOUR custom model on the same image

compare_url = 'https://assets.epicurious.com/photos/68557ec0c13b32a411b3972b/16:9/w_1920%2Cc_limit/types-of-apples_LEDE_V3_061025_3678_VOG_final.jpg'

if 'PASTE' not in compare_url:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Pre-trained COCO vs Your Custom Model', fontsize=14, fontweight='bold')

    for ax, model, title in [
        (axes[0], pretrained_model, 'Pre-trained YOLOv8 (COCO 80 classes)'),
        (axes[1], trained_model,    f'Your Custom Model ({len(trained_model.names)} classes)'),
    ]:
        res = model.predict(source=compare_url, conf=0.25, verbose=False)
        img = res[0].plot()[:, :, ::-1]
        ax.imshow(img)
        ax.set_title(title, fontsize=11)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

    print('\n💡 Key insight: the pre-trained model knows many things but misses your specific objects.')
    print('   Your custom model knows only your classes — but detects them properly.')
    print('   This is the power of transfer learning + fine-tuning.')
else:
    print('⚠️  Paste your test image URL above first.')

---
## Section 7 — 🟡 Go Further (If You're Ahead)

**Challenge A — Try a bigger model:**
Replace `yolov8n.pt` with `yolov8s.pt` (small) and retrain. Compare mAP. How much does accuracy improve? How much slower is training?

**Challenge B — Export for deployment:**
```python
# Export to ONNX (runs anywhere — no Python needed)
trained_model.export(format='onnx')

# Export to TensorFlow Lite (mobile deployment)
trained_model.export(format='tflite')
```

**Challenge C — Detect in a video:**
```python
from google.colab import files
uploaded = files.upload()   # upload a short .mp4
results = trained_model.predict(source='your_video.mp4', save=True)
# Output saved to runs/detect/predict/
```

**Challenge D — Add a new class:**
Go back to Roboflow. Add a new class and annotate 25+ more images. Generate a new version. Retrain. How does mAP change?

**Challenge E — Think about integration:**
Sketch on paper: how would you connect your detector to:
- An alert system (email/SMS when object detected)?
- A counter (how many objects passed through a zone)?
- A chatbot (describe what's in an image)?
You'll build something like this tomorrow.

In [ ]:
# ── Export your model ─────────────────────────────────────────────────────────
# Challenge B starter

# Export to ONNX (for deployment anywhere)
# trained_model.export(format='onnx')
# print('Exported to ONNX')

# Download the best weights to your computer
from google.colab import files
# files.download('workshop_runs/my_detector/weights/best.pt')

print('Uncomment the lines above to export or download your model.')

---
## Section 8 — Day 3 Recap

Make sure you can answer these before moving on:

1. **What does each layer of a CNN learn?**
   > *Layer 1: edges. Layer 2: shapes. Layer 3+: parts. Final layers: whole objects. All learned automatically from labelled examples.*

2. **What is transfer learning and why does it matter?**
   > *Starting from a model pre-trained on millions of images (COCO), so you only need to fine-tune on your specific objects. 30 images can be enough instead of millions.*

3. **What do box_loss and cls_loss measure?**
   > *box_loss = error in bounding box position/size. cls_loss = error in predicting the correct class. Both should decrease during training.*

4. **What is mAP@50?**
   > *Mean Average Precision at 50% IoU threshold. Overall quality score for your detector. Higher = better.*

5. **What is the difference between precision and recall?**
   > *Precision: of all detections made, % correct. Recall: of all real objects, % found. They trade off against each other via the confidence threshold.*

---

## 🚀 Coming Tomorrow — Day 4: Pipelines & Deployment

Tomorrow you'll:
- **Chain models together** — your detector + an LLM + an output layer
- **Build a Gradio interface** — a real web UI in under 10 lines
- **Deploy to HuggingFace Spaces** — a public URL anyone can use
- Understand system design: latency, accuracy, failure modes

> **Prep:** Make sure you have a HuggingFace account at huggingface.co

> **Think:** What would you connect your detector to? A chatbot? A counter? An alert?

**See you tomorrow. 🎯**